<p style="text-align:center">
    <a href="https://skills.network" target="_blank">
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/assets/logos/SN_web_lightmode.png" width="200" alt="Skills Network Logo">
    </a>
</p>


# **Space X  Falcon 9 First Stage Landing Prediction**


## Web scraping Falcon 9 and Falcon Heavy Launches Records from Wikipedia


Estimated time needed: **40** minutes


In this lab, you will be performing web scraping to collect Falcon 9 historical launch records from a Wikipedia page titled `List of Falcon 9 and Falcon Heavy launches`

https://en.wikipedia.org/wiki/List_of_Falcon_9_and_Falcon_Heavy_launches


![](https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/labs/module_1_L2/images/Falcon9_rocket_family.svg)


Falcon 9 first stage will land successfully


![](https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMDeveloperSkillsNetwork-DS0701EN-SkillsNetwork/api/Images/landing_1.gif)


Several examples of an unsuccessful landing are shown here:


![](https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMDeveloperSkillsNetwork-DS0701EN-SkillsNetwork/api/Images/crash.gif)


More specifically, the launch records are stored in a HTML table shown below:


![](https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/labs/module_1_L2/images/falcon9-launches-wiki.png)


  ## Objectives
Web scrap Falcon 9 launch records with `BeautifulSoup`: 
- Extract a Falcon 9 launch records HTML table from Wikipedia
- Parse the table and convert it into a Pandas data frame


First let's import required packages for this lab


In [1]:
# Install once in your environment if needed:
# python -m pip install beautifulsoup4 requests pandas
# Dependencies are imported in the next cell.

In [2]:
import re
import unicodedata
from pathlib import Path

import pandas as pd
import requests
from bs4 import BeautifulSoup

pd.set_option('display.max_columns', None)

and we will provide some helper functions for you to process web scraped HTML table


In [3]:
def date_time(table_cells):
    """
    This function returns the data and time from the HTML  table cell
    Input: the  element of a table data cell extracts extra row
    """
    return [data_time.strip() for data_time in list(table_cells.strings)][0:2]

def booster_version(table_cells):
    """
    This function returns the booster version from the HTML  table cell 
    Input: the  element of a table data cell extracts extra row
    """
    out=''.join([booster_version for i,booster_version in enumerate( table_cells.strings) if i%2==0][0:-1])
    return out

def landing_status(table_cells):
    """
    This function returns the landing status from the HTML table cell 
    Input: the  element of a table data cell extracts extra row
    """
    out=[i for i in table_cells.strings][0]
    return out


def get_mass(table_cells):
    mass=unicodedata.normalize("NFKD", table_cells.text).strip()
    if mass:
        mass.find("kg")
        new_mass=mass[0:mass.find("kg")+2]
    else:
        new_mass=0
    return new_mass


def extract_column_from_header(row):
    """
    This function returns the landing status from the HTML table cell 
    Input: the  element of a table data cell extracts extra row
    """
    if (row.br):
        row.br.extract()
    if row.a:
        row.a.extract()
    if row.sup:
        row.sup.extract()
        
    colunm_name = ' '.join(row.contents)
    
    # Filter the digit and empty names
    if not(colunm_name.strip().isdigit()):
        colunm_name = colunm_name.strip()
        return colunm_name    


To keep the lab tasks consistent, you will be asked to scrape the data from a snapshot of the  `List of Falcon 9 and Falcon Heavy launches` Wikipage updated on
`9th June 2021`


In [4]:
static_url = "https://en.wikipedia.org/w/index.php?title=List_of_Falcon_9_and_Falcon_Heavy_launches&oldid=1027686922"

headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                  "AppleWebKit/537.36 (KHTML, like Gecko) "
                  "Chrome/91.0.4472.124 Safari/537.36"
}

Next, request the HTML page from the above URL and get a `response` object


### TASK 1: Request the Falcon9 Launch Wiki page from its URL


First, let's perform an HTTP GET method to request the Falcon9 Launch HTML page, as an HTTP response.


In [5]:
response = requests.get(static_url, headers=headers, timeout=30)
response.raise_for_status()
print(f'Received {len(response.content):,} bytes with HTTP {response.status_code}.')

Received 3,273,603 bytes with HTTP 200.


Create a `BeautifulSoup` object from the HTML `response`


In [6]:
soup = BeautifulSoup(response.text, 'html.parser')

Print the page title to verify if the `BeautifulSoup` object was created properly 


In [7]:
print(soup.title.get_text(strip=True))

List of Falcon 9 and Falcon Heavy launches - Wikipedia


### TASK 2: Extract all column/variable names from the HTML table header


Next, we want to collect all relevant column names from the HTML table header


Let's try to find all tables on the wiki page first. If you need to refresh your memory about `BeautifulSoup`, please check the external reference link towards the end of this lab


In [8]:
html_tables = soup.find_all('table', class_='wikitable')
launch_tables = soup.select('table.wikitable.plainrowheaders.collapsible')
print(f'Wiki tables found: {len(html_tables)}; launch tables found: {len(launch_tables)}')

Wiki tables found: 13; launch tables found: 9


Starting from the third table is our target table contains the actual launch records.


In [9]:
if len(launch_tables) < 1:
    raise ValueError('No Falcon launch tables were found in the archived page.')
first_launch_table = launch_tables[0]
print(first_launch_table.prettify()[:2_000])

<table class="wikitable plainrowheaders collapsible" id="mwmQ" style="width: 100%;">
 <tbody id="mwmg">
  <tr id="mwmw">
   <th id="mwnA" scope="col">
    Flight No.
   </th>
   <th id="mwnQ" scope="col">
    Date and
    <br id="mwng"/>
    time (
    <a href="https://en.wikipedia.org/wiki/Coordinated_Universal_Time" id="mwnw" rel="mw:WikiLink" title="Coordinated Universal Time">
     UTC
    </a>
    )
   </th>
   <th id="mwoA" scope="col">
    <a href="https://en.wikipedia.org/wiki/List_of_Falcon_9_first-stage_boosters" id="mwoQ" rel="mw:WikiLink" title="List of Falcon 9 first-stage boosters">
     Version,
     <br id="mwog"/>
     Booster
    </a>
    <sup about="#mwt70" class="mw-ref reference" data-mw='{"name":"ref","attrs":{"name":"booster","group":"lower-alpha"},"body":{"id":"mw-reference-text-cite_note-booster-11"},"parts":[{"template":{"target":{"wt":"efn","href":"./Template:Efn"},"params":{"name":{"wt":"booster"},"1":{"wt":"Falcon 9 first-stage boosters are designated with 

You should able to see the columns names embedded in the table header elements `<th>` as follows:


```
<tr>
<th scope="col">Flight No.
</th>
<th scope="col">Date and<br/>time (<a href="/wiki/Coordinated_Universal_Time" title="Coordinated Universal Time">UTC</a>)
</th>
<th scope="col"><a href="/wiki/List_of_Falcon_9_first-stage_boosters" title="List of Falcon 9 first-stage boosters">Version,<br/>Booster</a> <sup class="reference" id="cite_ref-booster_11-0"><a href="#cite_note-booster-11">[b]</a></sup>
</th>
<th scope="col">Launch site
</th>
<th scope="col">Payload<sup class="reference" id="cite_ref-Dragon_12-0"><a href="#cite_note-Dragon-12">[c]</a></sup>
</th>
<th scope="col">Payload mass
</th>
<th scope="col">Orbit
</th>
<th scope="col">Customer
</th>
<th scope="col">Launch<br/>outcome
</th>
<th scope="col"><a href="/wiki/Falcon_9_first-stage_landing_tests" title="Falcon 9 first-stage landing tests">Booster<br/>landing</a>
</th></tr>
```


Next, we just need to iterate through the `<th>` elements and apply the provided `extract_column_from_header()` to extract column name one by one


In [10]:
column_names = []
for header in first_launch_table.find_all('th'):
    name = extract_column_from_header(header)
    if name is not None and len(name) > 0 and name not in column_names:
        column_names.append(name)

column_names

['Flight No.',
 'Date and time ( )',
 'Launch site',
 'Payload',
 'Payload mass',
 'Orbit',
 'Customer',
 'Launch outcome']

Check the extracted column names


In [11]:
print(column_names)

['Flight No.', 'Date and time ( )', 'Launch site', 'Payload', 'Payload mass', 'Orbit', 'Customer', 'Launch outcome']


## TASK 3: Create a data frame by parsing the launch HTML tables


We will create an empty dictionary with keys from the extracted column names in the previous task. Later, this dictionary will be converted into a Pandas dataframe


In [12]:
output_columns = [
    'Flight No.', 'Launch site', 'Payload', 'Payload mass', 'Orbit',
    'Customer', 'Launch outcome', 'Version Booster', 'Booster landing',
    'Date', 'Time',
]
launch_dict = {column: [] for column in output_columns}

Next, we just need to fill up the `launch_dict` with launch records extracted from table rows.


Usually, HTML tables in Wiki pages are likely to contain unexpected annotations and other types of noises, such as reference links `B0004.1[8]`, missing values `N/A [e]`, inconsistent formatting, etc.


To simplify the parsing process, we have provided an incomplete code snippet below to help you to fill up the `launch_dict`. Please complete the following code snippet with TODOs or you can choose to write your own logic to parse all launch tables:


In [13]:
def clean_cell(cell):
    """Return readable cell text without citation markers or extra whitespace."""
    for tag in cell.select('sup, span.reference'):
        tag.decompose()
    return re.sub(r'\s+', ' ', cell.get_text(' ', strip=True))


extracted_row = 0
for table in launch_tables:
    for row in table.find_all('tr'):
        row_header = row.find('th')
        cells = row.find_all('td')
        if row_header is None or len(cells) < 9:
            continue

        flight_match = re.fullmatch(r'\d+', clean_cell(row_header))
        if flight_match is None:
            continue

        date_and_time = date_time(cells[0])
        date = date_and_time[0].rstrip(',') if date_and_time else None
        time = date_and_time[1] if len(date_and_time) > 1 else None

        launch_dict['Flight No.'].append(flight_match.group())
        launch_dict['Date'].append(date)
        launch_dict['Time'].append(time)
        launch_dict['Version Booster'].append(clean_cell(cells[1]))
        launch_dict['Launch site'].append(clean_cell(cells[2]))
        launch_dict['Payload'].append(clean_cell(cells[3]))
        launch_dict['Payload mass'].append(get_mass(cells[4]))
        launch_dict['Orbit'].append(clean_cell(cells[5]))
        launch_dict['Customer'].append(clean_cell(cells[6]))
        launch_dict['Launch outcome'].append(clean_cell(cells[7]))
        launch_dict['Booster landing'].append(landing_status(cells[8]).strip())
        extracted_row += 1

print(f'Extracted {extracted_row} launch records.')

Extracted 121 launch records.


After you have fill in the parsed launch record values into `launch_dict`, you can create a dataframe from it.


In [14]:
df = pd.DataFrame(launch_dict)
assert not df.empty, 'No launch records were parsed.'
assert df.columns.tolist() == output_columns
assert df['Flight No.'].astype(str).str.fullmatch(r'\d+').all()

print('Shape:', df.shape)
print('Column names:', df.columns.tolist())
print('Missing-value counts:')
display(df.isna().sum().rename('missing').to_frame())
df.head()

Shape: (121, 11)
Column names: ['Flight No.', 'Launch site', 'Payload', 'Payload mass', 'Orbit', 'Customer', 'Launch outcome', 'Version Booster', 'Booster landing', 'Date', 'Time']
Missing-value counts:


,missing
Flight No.,0
Launch site,0
Payload,0
Payload mass,0
Orbit,0
Customer,0
Launch outcome,0
Version Booster,0
Booster landing,0
Date,0


,Flight No.,Launch site,Payload,Payload mass,Orbit,Customer,Launch outcome,Version Booster,Booster landing,Date,Time
0,1,"CCAFS , SLC-40",Dragon Spacecraft Qualification Unit,0,LEO,SpaceX,Success,F9 v1.0 B0003.1,Failure,4 June 2010,18:45
1,2,"CCAFS , SLC-40",Dragon demo flight C1 (Dragon C101),0,LEO ( ISS ),NASA ( COTS ) NRO,Success,F9 v1.0 B0004.1,Failure,8 December 2010,15:43
2,3,"CCAFS , SLC-40",Dragon demo flight C2+ (Dragon C102),525 kg,LEO ( ISS ),NASA ( COTS ),Success,F9 v1.0 B0005.1,No,22 May 2012,07:44
3,4,"CCAFS , SLC-40",SpaceX CRS-1 (Dragon C103),"4,700 kg",LEO ( ISS ),NASA ( CRS ),Success,F9 v1.0 B0006.1,No attempt,8 October 2012,00:35
4,5,"CCAFS , SLC-40",SpaceX CRS-2 (Dragon C104),"4,877 kg",LEO ( ISS ),NASA ( CRS ),Success,F9 v1.0 B0007.1,No,1 March 2013,15:10


We can now export it to a <b>CSV</b> for the next section, but to make the answers consistent and in case you have difficulties finishing this lab. 

Following labs will be using a provided dataset to make each lab independent. 


In [15]:
output_path = Path('spacex_web_scraped.csv')
df.to_csv(output_path, index=False)
print(f'Saved {len(df)} rows to {output_path.resolve()}')

Saved 121 rows to K:\Uni Coding\ibm-data-science-capstone\02-data-collection-web-scraping\spacex_web_scraped.csv


## Authors


<a href="https://www.linkedin.com/in/yan-luo-96288783/">Yan Luo</a>


<a href="https://www.linkedin.com/in/nayefaboutayoun/">Nayef Abou Tayoun</a>


<!--
## Change Log
-->


<!--
| Date (YYYY-MM-DD) | Version | Changed By | Change Description      |
| ----------------- | ------- | ---------- | ----------------------- |
| 2021-06-09        | 1.0     | Yan Luo    | Tasks updates           |
| 2020-11-10        | 1.0     | Nayef      | Created the initial version |
-->


Copyright © 2021 IBM Corporation. All rights reserved.
